# DINO dataset-fit exploration — checkpoints 1–2

We are studying **pretrained DINO patch features**, starting with DINOv2-B/14 and later repeating the same inputs with DINOv3-B/16. No model is trained in this study.

The team has already validated and inspected the dataset. Here we only prepare a fixed DINO study sample and its paired inputs; we do not rerun dataset-quality audits.

**Checkpoint 1:** environment → select matched trios → freeze 448×448 crops → preview one trio. **Checkpoint 2:** frozen DINOv2 → inspect spatial tokens → raw feature distances → shared-scale heatmaps for that trio. Run cells in order and pause at each section to understand its outputs.

Select the project's **Python 3.11 `.venv` kernel** first; setup commands are in [README.md](README.md). The kernel readout should report CPU on this laptop. Checkpoint 1 has no downloads; checkpoint 2 fetches pinned pretrained weights on its first run.

**Study design:** one regression per operator from the committed training split, with benign and no-change controls from the same page and at the same location. Locations are chosen using ground truth, so this is guided feature-sensitivity exploration, not full-page detection evaluation. One example per operator supports tentative observations only.

**Presentation relevance:** this checkpoint fixes the methodology and the exact inputs for the backbone comparison. It makes no claims about which model or mutation type is easier.


In [ ]:
# 1. Show exactly which environment will run the experiment.
# Expected: Python 3.11, torch 2.8.x+cpu, device=cpu on this laptop.
import importlib.metadata as metadata
import json
import platform
import sys
from pathlib import Path

import matplotlib
import numpy as np
import scipy
import torch
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
versions = {
    name: metadata.version(name)
    for name in ('torch', 'torchvision', 'transformers', 'numpy', 'scipy',
                 'Pillow', 'matplotlib', 'ipykernel')
}
environment = {
    'python': sys.version,
    'executable': sys.executable,
    'platform': platform.platform(),
    'packages': versions,
    'device': str(DEVICE),
    'torch_cuda_version': torch.version.cuda,
}
if DEVICE.type == 'cuda':
    environment['gpu'] = torch.cuda.get_device_name(DEVICE)
    environment['gpu_memory_gib'] = round(
        torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3, 2
    )
print(json.dumps(environment, indent=2))
print('NumPy, SciPy and PyTorch imports succeeded. No DINO model is loaded yet.')


In [1]:
# 2. Use existing metadata and the official split; never create a new split.
# Expected: 13 matched trios, each with regression/benign/no_change.
import random
from collections import defaultdict

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'pyproject.toml').is_file() and (p / 'src/regression_lens').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('Open the notebook from inside this repository.')

from regression_lens.data.records import load_records, operator_names
from regression_lens.data.splits import TEAM_SPLIT, load_split

DATA_ROOT = Path(r'D:\csc409-data\d2c_v3')  # Change this on the lab machine.
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'dino_fit'
SEED, CROP_SIZE = 490, 448
SPLIT_PATH = REPO_ROOT / TEAM_SPLIT
records = load_records(DATA_ROOT)
split = load_split(SPLIT_PATH, records)
train_pages = set(split['train'])
records_by_id = {r['sample_id']: r for r in records}
by_page = defaultdict(list)
for record in records:
    if record['parent_id'] in train_pages:
        by_page[record['parent_id']].append(record)

rng = random.Random(SEED)
trios, used_pages = [], set()
for index, operator in enumerate(operator_names(records)):
    candidates = sorted(
        (r for rows in by_page.values() for r in rows
         if r['label'] == 'regression' and r['operator'] == operator
         and all(any(c['label'] == label for c in by_page[r['parent_id']])
                 for label in ('benign', 'no_change'))),
        key=lambda r: r['sample_id'],
    )
    rng.shuffle(candidates)
    if not candidates:
        raise RuntimeError(f'No matched trio available for {operator}; do not silently omit it.')
    regression = next((r for r in candidates if r['parent_id'] not in used_pages), candidates[0])
    controls = by_page[regression['parent_id']]
    browser = ('firefox', 'webkit')[index % 2]
    benign = sorted(
        (r for r in controls if r['label'] == 'benign'),
        key=lambda r: (r.get('browser_cand') != browser, r['sample_id']),
    )[0]
    no_change = min((r for r in controls if r['label'] == 'no_change'),
                    key=lambda r: r['sample_id'])
    trios.append({'operator': operator, 'parent_id': regression['parent_id'],
                  'sample_ids': [r['sample_id'] for r in (regression, benign, no_change)]})
    used_pages.add(regression['parent_id'])

for trio in trios:
    print(f"{trio['operator']:18} {trio['parent_id']} | " + ', '.join(trio['sample_ids']))
print(f'\n{len(trios)} trios / {3 * len(trios)} pair-crop evaluations; train pages only.')


NameError: name 'Path' is not defined

In [ ]:
# 3. Freeze model-independent crop geometry, without resizing either image.
# Expected: a reusable sample_manifest.json under ignored outputs/dino_fit/.
import hashlib
import math
from PIL import Image

def crop_rectangle(regression):
    boxes = [regression.get(key) for key in ('bbox_before', 'bbox_after')]
    boxes = [box for box in boxes if box is not None]
    if not boxes:
        raise ValueError(f"{regression['sample_id']}: edited-element boxes unavailable")
    left = min(box[0] for box in boxes)
    top = min(box[1] for box in boxes)
    right = max(box[0] + box[2] for box in boxes)
    bottom = max(box[1] + box[3] for box in boxes)
    width, height = regression['size_ref']
    x = max(0, min(math.floor((left + right - CROP_SIZE) / 2), max(0, width - CROP_SIZE)))
    y = max(0, min(math.floor((top + bottom - CROP_SIZE) / 2), max(0, height - CROP_SIZE)))
    return [x, y, x + CROP_SIZE, y + CROP_SIZE]

def load_crop(relative_path, rectangle):
    # Every paired image uses this same original-coordinate rectangle.
    # Keep a validity mask so later statistics can ignore white padding.
    x0, y0, x1, y1 = rectangle
    canvas = Image.new('RGB', (x1 - x0, y1 - y0), 'white')
    valid = np.zeros((y1 - y0, x1 - x0), dtype=bool)
    with Image.open(DATA_ROOT / relative_path) as source:
        rgb = source.convert('RGB')
        right, bottom = min(x1, rgb.width), min(y1, rgb.height)
        if right > x0 and bottom > y0:
            canvas.paste(rgb.crop((x0, y0, right, bottom)), (0, 0))
            valid[:bottom - y0, :right - x0] = True
    return canvas, valid

for trio in trios:
    trio['crop_xyxy'] = crop_rectangle(records_by_id[trio['sample_ids'][0]])
manifest = {
    'format_version': 1, 'purpose': 'ground-truth-guided DINO feature-fit exploration',
    'seed': SEED, 'subset': 'train', 'crop_size': CROP_SIZE,
    'split_sha256': hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest(),
    'metadata_sha256': hashlib.sha256(
        (DATA_ROOT / 'master_dataset_with_benign.json').read_bytes()).hexdigest(),
    'trios': trios,
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_ROOT / 'sample_manifest.json'
if manifest_path.exists():
    if json.loads(manifest_path.read_text(encoding='utf-8')) != manifest:
        raise RuntimeError('Existing study differs. Preserve it and choose a new OUTPUT_ROOT.')
else:
    manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
(OUTPUT_ROOT / 'environment.json').write_text(
    json.dumps(environment, indent=2) + '\n', encoding='utf-8')
print(f'Frozen sample: {manifest_path}')
print('Crop size:', CROP_SIZE, 'x', CROP_SIZE, '; resize: none; missing image area: white padding.')


In [ ]:
# 4. Preview exactly what DINO will receive for one matched trio.
# This is a model-input preview, not another general dataset inspection.
# Orange/cyan: edited element before/after. Red dashed: diff_bbox.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

PREVIEW_OPERATOR = 'font_size'  # Later try another operator from the printed list.
trio = next(t for t in trios if t['operator'] == PREVIEW_OPERATOR)
regression, benign, no_change = [records_by_id[sid] for sid in trio['sample_ids']]
rectangle = trio['crop_xyxy']
image_specs = [
    (regression, 'reference_path', 'Reference'),
    (regression, 'candidate_path', 'Regression'),
    (benign, 'candidate_path', f"Benign ({benign.get('browser_cand', 'browser')})"),
    (no_change, 'candidate_path', 'No change'),
]
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, (record, path_key, title) in zip(axes, image_specs):
    image, valid = load_crop(record[path_key], rectangle)
    ax.imshow(image)
    ax.set_title(f'{title}\n{valid.mean():.1%} real image area')
    ax.axis('off')
    boxes = []
    if title == 'Reference':
        boxes = [('bbox_before', 'orange', '-')]
    elif record['label'] == 'regression':
        boxes = [('bbox_after', 'cyan', '-'), ('diff_bbox', 'red', '--')]
    for key, color, style in boxes:
        if record.get(key) is not None:
            x, y, width, height = record[key]
            ax.add_patch(Rectangle((x - rectangle[0], y - rectangle[1]), width, height,
                                   fill=False, edgecolor=color, linestyle=style, linewidth=2))
    ax.set_xlim(-0.5, CROP_SIZE - 0.5)
    ax.set_ylim(CROP_SIZE - 0.5, -0.5)
fig.suptitle(f"{PREVIEW_OPERATOR} | {trio['parent_id']} | crop {rectangle}")
fig.tight_layout()
preview_path = OUTPUT_ROOT / f'input_trio_{PREVIEW_OPERATOR}.png'
fig.savefig(preview_path, dpi=150, bbox_inches='tight')
plt.show()
print('Pair labels:', [r['label'] for r in (regression, benign, no_change)])
print('Crop image shape: (448, 448, 3); next checkpoint makes [1, 3, 448, 448] tensors.')
print(f'Preview saved to: {preview_path}')
print('Checkpoint 1 complete. Continue to checkpoint 2 once the crop and controls are understood.')


## Checkpoint 2 — frozen DINOv2 on one matched trio

Run these four code cells after checkpoint 1. The first run downloads the official `facebook/dinov2-base` checkpoint into ignored `outputs/dino_fit/model_cache/`; later runs reuse it. The model revision is pinned. No optimizer or training is involved.

**Shapes:** each input is `[1, 3, 448, 448]`. DINOv2-B/14 divides it into `32 × 32 = 1024` patches. Its output is `[1, 1025, 768]`: one CLS token plus 1024 spatial tokens, each a 768-dimensional descriptor. We compare spatial tokens only.

**Distances:** cosine measures direction change; raw L2 also responds to feature magnitude. Both use final, layer-normalized backbone features. They are feature distances, not regression probabilities. We preserve native 32×32 values and upsample only for plotting.

**Inspect:** is the regression response near the red changed-region box? Does the benign browser variation activate more strongly? Is no-change essentially zero? Every panel for one distance shares a color scale; its upper limit is the pooled 99th percentile, so saturated outliers remain available in raw statistics. ROI means use fractional patch overlap with the matched regression region. Padding-only or partly padded patches are excluded from statistics.

**Presentation relevance:** these figures show the architecture and an actual preliminary example. A single guided crop cannot establish an operator-wide result or full-page detection performance. Stop after inspecting this trio before evaluating the remaining operators.


In [ ]:
# 5. Load pretrained weights, freeze all parameters, and retain float32 precision.
from transformers import AutoImageProcessor, AutoModel
from time import perf_counter

MODEL_ID = 'facebook/dinov2-base'
MODEL_REVISION = 'f9e44c814b77203eaa57a6bdbbd535f21ede1415'
MODEL_CACHE = OUTPUT_ROOT / 'model_cache'
if DEVICE.type == 'cpu':
    torch.set_num_threads(4)  # This laptop has four physical CPU cores.

processor = AutoImageProcessor.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, cache_dir=MODEL_CACHE,
    token=False, use_fast=False,
)
backbone = AutoModel.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, cache_dir=MODEL_CACHE,
    token=False, use_safetensors=True, attn_implementation='sdpa',
).to(device=DEVICE, dtype=torch.float32)
backbone.requires_grad_(False)
backbone.eval()
PATCH_SIZE = backbone.config.patch_size
GRID_SIZE = CROP_SIZE // PATCH_SIZE
assert PATCH_SIZE == 14 and GRID_SIZE == 32
assert not backbone.training and not any(p.requires_grad for p in backbone.parameters())
print(type(backbone).__name__, '| patch:', PATCH_SIZE, '| grid:', GRID_SIZE, 'x', GRID_SIZE)
print('Parameters:', sum(p.numel() for p in backbone.parameters()), '| trainable: 0')
print('Normalization:', processor.image_mean, processor.image_std)
print('Resize and center-crop will be disabled; pixel geometry stays exactly 448 x 448.')


In [ ]:
# 6. Inspect one forward pass, then use the SAME backbone for all three candidates.
crop_images, crop_validity = {}, {}
crop_images['reference'], crop_validity['reference'] = load_crop(
    regression['reference_path'], rectangle)
reference_input = processor(
    images=crop_images['reference'], return_tensors='pt',
    do_resize=False, do_center_crop=False,
)['pixel_values'].to(DEVICE)
assert tuple(reference_input.shape) == (1, 3, 448, 448)
if DEVICE.type == 'cuda':
    torch.cuda.synchronize()
started = perf_counter()
with torch.inference_mode():
    reference_output = backbone(pixel_values=reference_input)
    features_ref = reference_output.last_hidden_state[:, 1:, :].cpu()
reference_seconds = perf_counter() - started
print('Input:', tuple(reference_input.shape), '= batch, RGB, height, width')
print('Output fields:', list(reference_output.keys()))
print('All tokens:', tuple(reference_output.last_hidden_state.shape), '= batch, CLS+patches, channels')
print('CLS:', tuple(reference_output.last_hidden_state[:, 0, :].shape))
print('Spatial tokens:', tuple(features_ref.shape), '= batch, 1024 patches, 768 channels')
assert tuple(features_ref.shape) == (1, 1024, 768)
features = {'reference': features_ref}
forward_seconds = {'reference': reference_seconds}
for record in (regression, benign, no_change):
    label = record['label']
    assert record['reference_path'] == regression['reference_path']
    crop_images[label], crop_validity[label] = load_crop(record['candidate_path'], rectangle)
    inputs = processor(images=crop_images[label], return_tensors='pt',
                       do_resize=False, do_center_crop=False)['pixel_values'].to(DEVICE)
    assert tuple(inputs.shape) == tuple(reference_input.shape)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    started = perf_counter()
    with torch.inference_mode():
        features[label] = backbone(pixel_values=inputs).last_hidden_state[:, 1:, :].cpu()
    forward_seconds[label] = perf_counter() - started
    assert features[label].shape == features_ref.shape
    print(f'{label:12} spatial features: {tuple(features[label].shape)} | {forward_seconds[label]:.3f} s')
print('Forward timings exclude checkpoint downloads and image preprocessing.')


In [ ]:
# 7. Preserve native patch distances and raw summaries; do not normalize each map.
import torch.nn.functional as F
from regression_lens.evaluation.metrics import peak_point, point_in_box

def distance_maps(reference_features, candidate_features):
    return {
        'cosine': (1 - F.cosine_similarity(reference_features, candidate_features, dim=-1))
                  .clamp(0, 2).reshape(GRID_SIZE, GRID_SIZE).numpy(),
        'l2': torch.linalg.vector_norm(reference_features - candidate_features, dim=-1)
              .reshape(GRID_SIZE, GRID_SIZE).numpy(),
    }

def roi_patch_overlap(box):
    # Fraction of each native patch inside a box in original screenshot coordinates.
    x, y, width, height = box
    edges = np.arange(GRID_SIZE) * PATCH_SIZE
    xs, ys = edges + rectangle[0], edges + rectangle[1]
    overlap_x = np.maximum(0, np.minimum(xs + PATCH_SIZE, x + width) - np.maximum(xs, x))
    overlap_y = np.maximum(0, np.minimum(ys + PATCH_SIZE, y + height) - np.maximum(ys, y))
    return np.outer(overlap_y, overlap_x) / PATCH_SIZE**2

def weighted_mean(values, weights):
    return float(np.sum(values * weights) / weights.sum()) if weights.sum() > 0 else None

# Shared scorer convention: point at the edited element for reflow, otherwise diff_bbox.
roi = regression['bbox_after'] if regression['reflow'] else regression['diff_bbox']
roi_overlap = roi_patch_overlap(roi)
maps, valid_patches, statistics = {}, {}, []
for record in (regression, benign, no_change):
    label = record['label']
    maps[label] = distance_maps(features_ref, features[label])
    pixels_valid = crop_validity['reference'] & crop_validity[label]
    valid = pixels_valid.reshape(GRID_SIZE, PATCH_SIZE, GRID_SIZE, PATCH_SIZE).all(axis=(1, 3))
    if not valid.any():
        raise ValueError(f'No fully valid patches for {label}')
    valid_patches[label] = valid
    for distance, values in maps[label].items():
        raw = values[valid]
        if not np.isfinite(raw).all():
            raise ValueError(f'Non-finite {distance} distances for {label}')
        # A numerical-noise-only map has no meaningful localization peak.
        has_signal = (maps[label]['cosine'][valid].max() > 1e-6
                      or maps[label]['l2'][valid].max() > 1e-5)
        local_peak = peak_point(np.where(valid, values, -np.inf), PATCH_SIZE) if has_signal else None
        point = ([local_peak[0] + rectangle[0], local_peak[1] + rectangle[1]]
                 if local_peak is not None else None)
        row = {
            'sample_id': record['sample_id'], 'label': label, 'distance': distance,
            'valid_patches': int(valid.sum()),
            'min': float(raw.min()), 'mean': float(raw.mean()), 'std': float(raw.std()),
            'median': float(np.median(raw)), 'p95': float(np.quantile(raw, 0.95)),
            'max': float(raw.max()),
            'top5_mean': float(np.sort(raw)[-max(1, math.ceil(0.05 * len(raw))):].mean()),
            'matched_roi_mean': weighted_mean(values, roi_overlap * valid),
            'outside_roi_mean': weighted_mean(values, (1 - roi_overlap) * valid),
            'point_original_xy': point,
            'pointing_hit': (point_in_box(point, roi)
                             if label == 'regression' and point is not None else None),
        }
        statistics.append(row)
        print(f"{label:12} {distance:6} mean={row['mean']:.6g} "
              f"p95={row['p95']:.6g} max={row['max']:.6g} | "
              f"ROI={row['matched_roi_mean']} outside={row['outside_roi_mean']} "
              f"| regression pointing={row['pointing_hit']}")

# Consistency checks for our distance calculation, not a repeat dataset audit.
self_maps = distance_maps(features_ref, features_ref)
assert self_maps['l2'].max() == 0 and self_maps['cosine'].max() < 1e-6
print('Self-comparison: L2 exactly zero; cosine within float32 roundoff.')


In [ ]:
# 8. Shared-scale figures and reproducible raw artifacts for this trio only.
import subprocess
RESULT_ROOT = OUTPUT_ROOT / 'dinov2' / PREVIEW_OPERATOR
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
display_limits = {}
labels = ('regression', 'benign', 'no_change')
for distance in ('cosine', 'l2'):
    pooled = np.concatenate([maps[label][distance][valid_patches[label]] for label in labels])
    vmax = max(float(np.quantile(pooled, 0.99)), 1e-6)
    display_limits[distance] = {'vmin': 0.0, 'vmax': vmax, 'rule': 'pooled trio p99'}
    figure, axes = plt.subplots(3, 3, figsize=(12, 12), layout='constrained')
    for row_index, label in enumerate(labels):
        reference_ax, candidate_ax, heatmap_ax = axes[row_index]
        reference_ax.imshow(crop_images['reference'])
        candidate_ax.imshow(crop_images[label])
        expanded = F.interpolate(torch.from_numpy(maps[label][distance]).view(1, 1, GRID_SIZE, GRID_SIZE),
                                 size=(CROP_SIZE, CROP_SIZE), mode='bilinear',
                                 align_corners=False)[0, 0].numpy()
        valid_cells_expanded = valid_patches[label].repeat(PATCH_SIZE, 0).repeat(PATCH_SIZE, 1)
        real_pixels = crop_validity['reference'] & crop_validity[label] & valid_cells_expanded
        heat_image = heatmap_ax.imshow(np.ma.masked_where(~real_pixels, expanded),
                                       cmap='magma', vmin=0, vmax=vmax)
        for ax in (candidate_ax, heatmap_ax):
            x, y, width, height = roi
            ax.add_patch(Rectangle((x - rectangle[0], y - rectangle[1]), width, height,
                                   fill=False, edgecolor='lime', linestyle='--', linewidth=1.5))
        if label == 'regression':
            x, y, width, height = regression['diff_bbox']
            candidate_ax.add_patch(Rectangle((x - rectangle[0], y - rectangle[1]), width, height,
                                            fill=False, edgecolor='red', linewidth=1.5))
        summary = next(s for s in statistics if s['label'] == label and s['distance'] == distance)
        if summary['point_original_xy'] is not None:
            px, py = summary['point_original_xy']
            heatmap_ax.scatter(px - rectangle[0], py - rectangle[1], c='cyan', marker='+', s=60)
        reference_ax.set_title('Reference')
        candidate_ax.set_title(f'{label}: {PREVIEW_OPERATOR if label == "regression" else "control"}')
        heatmap_ax.set_title(f"{distance}: mean {summary['mean']:.3g}, max {summary['max']:.3g}")
        for ax in axes[row_index]:
            ax.set_xlim(-0.5, CROP_SIZE - 0.5)
            ax.set_ylim(CROP_SIZE - 0.5, -0.5)
            ax.axis('off')
    figure.colorbar(heat_image, ax=axes[:, 2], shrink=0.75, extend='max',
                    label=f'Raw {distance} distance; one shared scale')
    figure.suptitle(f'DINOv2-B/14 | {PREVIEW_OPERATOR} | native 32 x 32 patches\n'
                   'Green: matched regression ROI; red: regression diff_bbox; cyan: peak')
    figure.savefig(RESULT_ROOT / f'{distance}_heatmaps.png', dpi=150)
    plt.show()

np.savez_compressed(
    RESULT_ROOT / 'native_maps.npz',
    **{f'{label}_{distance}': values for label, distances in maps.items()
       for distance, values in distances.items()},
    **{f'{label}_valid_patches': valid_patches[label] for label in labels},
)
report = {
    'model_id': MODEL_ID, 'revision': MODEL_REVISION,
    'environment': environment, 'device': str(DEVICE), 'dtype': 'float32',
    'attention': 'sdpa', 'cpu_threads': torch.get_num_threads(),
    'sample_manifest_sha256': hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    'notebook_sha256': hashlib.sha256(
        (REPO_ROOT / 'notebooks/dino_exploration.ipynb').read_bytes()).hexdigest(),
    'git_commit': subprocess.check_output(
        ['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip(),
    'working_tree_dirty': bool(subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=REPO_ROOT, text=True).strip()),
    'sample_ids': trio['sample_ids'], 'crop_xyxy': rectangle,
    'patch_size': PATCH_SIZE, 'roi_original_xywh': roi,
    'roi_source_sample_id': regression['sample_id'],
    'roi_rule': 'bbox_after for reflow, otherwise diff_bbox',
    'normalization_mean': processor.image_mean, 'normalization_std': processor.image_std,
    'resize': False, 'center_crop': False,
    'forward_seconds': forward_seconds, 'display_limits': display_limits,
    'statistics': statistics,
}
(RESULT_ROOT / 'statistics.json').write_text(
    json.dumps(report, indent=2, allow_nan=False) + '\n', encoding='utf-8')
print(f'Figures, native maps and raw statistics saved to: {RESULT_ROOT}')
print('STOP: compare the regression and browser-control responses before running other operators.')
